In [1]:
# ============================================================
# NB 01. 라이브러리 설치 & 데이터 로드 & 전처리
# Brain Tumor Survival Analysis — Explainable AI
#
# 변수명 설계:
#   수치형  : Age, Sex, Prior_cancer, Treatment
#   범주형  : Age_group, Grade, Site, Era, Tx_group
#   인코딩  : Age_Young, Age_Middle, Age_Old
#             Grade_G2, Grade_G3, Grade_G4
#             Site_BrainNOS, Site_Cerebrum, Site_Other
#             Era_After2005, Era_Before2005
#             Tx_Single, Tx_Standard, Tx_None
#   생존    : time, event
# ============================================================

# ============================================================
# STEP 0. 라이브러리 설치
# ============================================================

import subprocess, sys

def install(package):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", package, "-q"]
    )

install("lifelines")
install("scikit-survival")
install("shap")
install("survshap")
install("pycox")
install("joblib")

print("✓ 라이브러리 설치 완료")

# ============================================================
# STEP 1. 라이브러리 import & 기본 설정
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
import os
import joblib

warnings.filterwarnings('ignore')

import platform
system = platform.system()
if system == 'Windows':
    font = 'Malgun Gothic'
elif system == 'Darwin':
    font = 'AppleGothic'
else:
    subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'],
                   capture_output=True)
    import matplotlib.font_manager as fm
    fm._load_fontmanager(try_read_cache=False)
    font = 'NanumGothic'

matplotlib.rcParams.update({
    'font.family'      : font,
    'axes.unicode_minus': False,
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

DATA_PATH  = "clinical.tsv"
OUTPUT_DIR = "./outputs/"
MODEL_DIR  = "./outputs/models/"
SEED       = 42

for d in [OUTPUT_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

np.random.seed(SEED)
print(f"✓ 환경 설정 완료 (폰트: {font})")

# ============================================================
# STEP 2. 원본 로드 & 기본 필터링
# ============================================================

df_raw = pd.read_csv(DATA_PATH, sep="\t")
df     = df_raw[df_raw['cases.index_date'] == 'Diagnosis'].copy()

MISSING_TOKENS = ["'--", "--", "not reported", "unknown", "Unknown"]
df.replace(MISSING_TOKENS, np.nan, inplace=True)

print(f"원본 shape        : {df_raw.shape}")
print(f"필터링 후 shape   : {df.shape}")

# ============================================================
# STEP 3. 생존 변수 정의
#   event : 1 = Dead, 0 = Alive (censored)
#   time  : 생존 기간 (days)
# ============================================================

df['event'] = (
    df['demographic.vital_status']
    .map({'Dead': 1, 'Alive': 0})
    .astype(float)
)

df['time'] = np.where(
    df['event'] == 1,
    pd.to_numeric(df['demographic.days_to_death'],        errors='coerce'),
    pd.to_numeric(df['diagnoses.days_to_last_follow_up'], errors='coerce')
)

df = df.dropna(subset=['event', 'time'])
df = df[df['time'] > 0]
df['event'] = df['event'].astype(int)

# ============================================================
# STEP 4. 원발성 종양 필터링 & 중복 제거
# ============================================================

df = df[df['diagnoses.classification_of_tumor'] == 'primary'].copy()

# 치료 조합 집계 (중복 제거 전에 먼저 수행)
treatment_agg = (
    df.groupby('cases.submitter_id')['treatments.treatment_type']
    .apply(lambda x: '+'.join(sorted(set(
        str(v) for v in x if pd.notna(v)
    ))))
    .reset_index()
    .rename(columns={'treatments.treatment_type': 'full_treatment_path'})
)

# 환자당 최장 추적 관찰 기간 1건 유지
df = (
    df.sort_values('time', ascending=False)
      .drop_duplicates(subset='cases.submitter_id', keep='first')
      .reset_index(drop=True)
)

df = df.merge(treatment_agg, on='cases.submitter_id', how='left')
print(f"\n중복 제거 후 인원 : {len(df):,}명")

# ============================================================
# STEP 5. Age — 연령
#   Age      : 수치형 (세)
#   Age_group: Young(0-39) / Middle(40-59) / Old(60+)
#   근거: WHO CNS Tumor 분류 및 TCGA 기반 뇌종양 연구 표준
# ============================================================

df['Age'] = pd.to_numeric(
    df['demographic.age_at_index'], errors='coerce'
)
missing_age = df['Age'].isnull().sum()
print(f"Age 결측치: {missing_age}명 ({missing_age/len(df)*100:.1f}%)")
# 결측치가 5% 미만이면 중앙값 대체 근거 있음
df['Age'].fillna(df['Age'].median(), inplace=True)

df['Age_group'] = pd.cut(
    df['Age'],
    bins=[0, 40, 60, 120],
    labels=['Young', 'Middle', 'Old'],
    right=False
)

# ============================================================
# STEP 6. Sex — 성별
#   1 = Male, 0 = Female
# ============================================================

df['Sex'] = (
    df['demographic.gender'].str.lower()
    .map({'male': 1, 'female': 0})
)

# ============================================================
# STEP 7. Grade — 종양 등급 (WHO 기준)
#   우선순위: morphology 코드 → anaplastic 진단명 → tumor_grade
#   G4: 교모세포종 계열
#   G3: anaplastic 계열
#   G2: 저등급 신경교종
# ============================================================

def assign_grade(row):
    morph = str(row['diagnoses.morphology'])
    diag  = str(row['diagnoses.primary_diagnosis']).lower()
    grade = str(row['diagnoses.tumor_grade'])

    if morph in ['9680/3', '8720/3']:
        return np.nan                              # 비뇌종양 제외
    if morph in ['9440/3', '9442/3', '9474/3']:
        return 'G4'
    if grade == 'G3' or 'anaplastic' in diag:
        return 'G3'
    if grade == 'G2' or morph in ['9400/3', '9450/3', '9382/3']:
        return 'G2'
    return np.nan

df['Grade'] = df.apply(assign_grade, axis=1)

before = len(df)
df = df.dropna(subset=['Grade']).copy()
print(f"Grade 분류 불가 제거: {before - len(df)}명 → 최종 {len(df):,}명")

# ============================================================
# STEP 8. Site — 발생 부위
#   Brain_NOS / Cerebrum / Other_Specific
# ============================================================

def assign_site(loc):
    loc = str(loc)
    if 'Brain, NOS' in loc: return 'Brain_NOS'
    if 'Cerebrum'   in loc: return 'Cerebrum'
    if 'Skin'       in loc: return np.nan
    return 'Other_Specific'

df['Site'] = df['diagnoses.tissue_or_organ_of_origin'].apply(assign_site)
df = df.dropna(subset=['Site']).copy()

# ============================================================
# STEP 9. Era — 진단 시대
#   After_2005 / Before_2005
# Era 기준: 2005년 Stupp et al. NEJM 발표로
# Temozolomide 병용 방사선치료가 교모세포종
# 표준치료로 확립됨 → 치료 패러다임 변화 기준점
# ============================================================

df['diag_year'] = pd.to_numeric(
    df['diagnoses.year_of_diagnosis'], errors='coerce'
)
df['diag_year'].fillna(df['diag_year'].median(), inplace=True)

df['Era'] = np.where(
    df['diag_year'] < 2005, 'Before_2005', 'After_2005'
)

# ============================================================
# STEP 10. Prior_cancer — 과거 암 이력
#   1 = Yes, 0 = No
# ============================================================

df['Prior_cancer'] = (
    df['diagnoses.prior_malignancy'].str.lower() == 'yes'
).astype(int)

# ============================================================
# STEP 11. Tx_group / Treatment — 치료 변수
#   Tx_group  : Standard / Single / Supportive_None
#   Treatment : 1 = 치료받음, 0 = 치료 없음 (이진)
# ============================================================

def assign_tx_combo(path):
    path = str(path).lower()
    has_rt    = 'radiation'    in path or 'beam'           in path
    has_chemo = 'chemotherapy' in path or 'pharmaceutical' in path
    if has_rt and has_chemo: return 'RT+Chemo'
    if has_rt:               return 'RT_Only'
    if has_chemo:            return 'Chemo_Only'
    return 'Supportive_Other'

def assign_tx_group(combo):
    if combo == 'RT+Chemo':                return 'Standard'
    if combo in ['RT_Only', 'Chemo_Only']: return 'Single'
    return 'Supportive_None'

df['tx_combo']  = df['full_treatment_path'].apply(assign_tx_combo)
df['Tx_group']  = df['tx_combo'].apply(assign_tx_group)
df['Treatment'] = (df['Tx_group'] != 'Supportive_None').astype(int)

# ============================================================
# STEP 12. 최종 데이터셋 구성 (df_survival)
# ============================================================

SELECTED_COLS = [
    'cases.submitter_id',
    'Age', 'Age_group', 'Sex',    # 인구통계
    'Grade', 'Site',               # 병리 (Hist_group 제외 — Grade와 중복)
    'Era', 'Prior_cancer',         # 임상
    'Treatment', 'Tx_group',       # 치료
    'time', 'event'                # 생존 변수
]

df_survival = (
    df[SELECTED_COLS]
    .copy()
    .set_index('cases.submitter_id')
)

CAT_COLS = ['Age_group', 'Grade', 'Site', 'Era', 'Tx_group']
df_survival[CAT_COLS] = df_survival[CAT_COLS].astype('category')

# ============================================================
# STEP 13. 전처리 결과 검증
# ============================================================

print("\n" + "="*50)
print(f"최종 코호트        : {len(df_survival):,}명")
print(f"사망 (event=1)     : {df_survival['event'].sum():,}명 "
      f"({df_survival['event'].mean():.1%})")
print(f"중도절단 (event=0) : {(df_survival['event']==0).sum():,}명")
print(f"생존기간 중앙값    : {df_survival['time'].median():.0f}일")
print("="*50)

print("\n── Grade 분포 ──")
print(df_survival['Grade'].value_counts())

print("\n── Tx_group 분포 ──")
print(df_survival['Tx_group'].value_counts())

print("\n── 결측치 확인 ──")
missing = df_survival.isnull().sum()
print(missing[missing > 0] if missing.any() else "결측치 없음 ✓")

# ============================================================
# STEP 14. df_survival 저장
# ============================================================

df_survival.to_csv(OUTPUT_DIR + "df_survival.csv", encoding='utf-8-sig')
joblib.dump(df_survival, MODEL_DIR + "df_survival.pkl")

print(f"\n✓ 전처리 완료")
print(f"  CSV : {OUTPUT_DIR}df_survival.csv")
print(f"  PKL : {MODEL_DIR}df_survival.pkl")

# ============================================================
# STEP 15. One-hot encoding & Train/Test Split 저장
#
# 기준 범주 (drop_first=False → VIF 검정 후 NB03에서 처리):
#   Age_group  → Young / Middle / Old
#   Grade      → G2 / G3 / G4
#   Site       → Brain_NOS / Cerebrum / Other_Specific
#   Era        → After_2005 / Before_2005
#   Tx_group   → Single / Standard / Supportive_None
#
# 더미 변수명:
#   Age_Young, Age_Middle, Age_Old
#   Grade_G2, Grade_G3, Grade_G4
#   Site_BrainNOS, Site_Cerebrum, Site_Other
#   Era_After2005, Era_Before2005
#   Tx_Single, Tx_Standard, Tx_None
# ============================================================

from sksurv.util import Surv
from sklearn.model_selection import train_test_split

CAT_ENCODE_COLS = ['Age_group', 'Grade', 'Site', 'Era', 'Tx_group']

df_encoded = pd.get_dummies(
    df_survival,
    columns=CAT_ENCODE_COLS,
    drop_first=False    # VIF 검정 후 NB03에서 기준 범주 결정
)

# bool → float
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(float)

# 더미 변수명 단순화
RENAME_MAP = {
    'Age_group_Young'         : 'Age_Young',
    'Age_group_Middle'        : 'Age_Middle',
    'Age_group_Old'           : 'Age_Old',
    'Grade_G2'                : 'Grade_G2',
    'Grade_G3'                : 'Grade_G3',
    'Grade_G4'                : 'Grade_G4',
    'Site_Brain_NOS'          : 'Site_BrainNOS',
    'Site_Cerebrum'           : 'Site_Cerebrum',
    'Site_Other_Specific'     : 'Site_Other',
    'Era_After_2005'          : 'Era_After2005',
    'Era_Before_2005'         : 'Era_Before2005',
    'Tx_group_Single'         : 'Tx_Single',
    'Tx_group_Standard'       : 'Tx_Standard',
    'Tx_group_Supportive_None': 'Tx_None',
}

df_encoded   = df_encoded.rename(columns=RENAME_MAP)
FEATURE_COLS = [
    RENAME_MAP.get(c, c)
    for c in df_encoded.columns
    if c not in ['time', 'event']
]

# sksurv Surv 객체
X = df_encoded[FEATURE_COLS].astype(float)
y = Surv.from_arrays(
    event = df_encoded['event'].astype(bool),
    time  = df_encoded['time'].astype(float)
)

# Train / Test split (80:20, event 기준 층화)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED,
    stratify=df_encoded['event']
)

# lifelines용 DataFrame
df_train = X_train.copy()
df_train['time']  = [t for _, t in y_train]
df_train['event'] = [int(e) for e, _ in y_train]

df_test = X_test.copy()
df_test['time']  = [t for _, t in y_test]
df_test['event'] = [int(e) for e, _ in y_test]

# 저장
joblib.dump({
    'X_train'     : X_train,
    'X_test'      : X_test,
    'y_train'     : y_train,
    'y_test'      : y_test,
    'df_train'    : df_train,
    'df_test'     : df_test,
    'feature_cols': FEATURE_COLS,
}, MODEL_DIR + "data_splits.pkl")

print(f"\nTrain : {len(X_train):,}명  |  Test : {len(X_test):,}명")
print(f"피처 수 : {len(FEATURE_COLS)}개")
print(f"event 비율 — Train : {df_train['event'].mean():.1%}  "
      f"/ Test : {df_test['event'].mean():.1%}")

print(f"\n피처 목록 ({len(FEATURE_COLS)}개):")
for col in FEATURE_COLS:
    print(f"  {col}")

print(f"\n✓ data_splits.pkl 저장 완료")
print("  → NB 03 이후 모든 모델에서 동일한 split 사용")

✓ 라이브러리 설치 완료
✓ 환경 설정 완료 (폰트: AppleGothic)
원본 shape        : (9990, 210)
필터링 후 shape   : (7981, 210)

중복 제거 후 인원 : 1,166명
Age 결측치: 61명 (5.2%)
Grade 분류 불가 제거: 3명 → 최종 1,163명

최종 코호트        : 1,163명
사망 (event=1)     : 659명 (56.7%)
중도절단 (event=0) : 504명
생존기간 중앙값    : 486일

── Grade 분포 ──
Grade
G4    651
G3    264
G2    248
Name: count, dtype: int64

── Tx_group 분포 ──
Tx_group
Standard           1076
Single               62
Supportive_None      25
Name: count, dtype: int64

── 결측치 확인 ──
결측치 없음 ✓

✓ 전처리 완료
  CSV : ./outputs/df_survival.csv
  PKL : ./outputs/models/df_survival.pkl

Train : 930명  |  Test : 233명
피처 수 : 18개
event 비율 — Train : 56.7%  / Test : 56.7%

피처 목록 (18개):
  Age
  Sex
  Prior_cancer
  Treatment
  Age_Young
  Age_Middle
  Age_Old
  Grade_G2
  Grade_G3
  Grade_G4
  Site_BrainNOS
  Site_Cerebrum
  Site_Other
  Era_After2005
  Era_Before2005
  Tx_Single
  Tx_Standard
  Tx_None

✓ data_splits.pkl 저장 완료
  → NB 03 이후 모든 모델에서 동일한 split 사용
